# BSD10k Confidence v5 Pipeline

이 노트북은 현재 프로젝트 디렉토리 구조를 기준으로 `confidence_model_v5_experiments.py`를 실행하는 v5 실험용 워크북입니다.

확인된 구조:

- 프로젝트 루트: `Dcase baseline`
- BSD10k metadata: `data/metadata/BSD10k_metadata.csv`
- BSD35k-CS metadata: `data/metadata/BSD35k-CS_metadata.csv`
- BSD10k audio/text embeddings: `data/features/clap_audio_embeddings`, `data/features/clap_text_embeddings`
- BSD35k-CS audio/text embeddings: `data/features/BSD35k_clap_audio_embeddings`, `data/features/BSD35k-CS_clap_text_embeddings`
- v5 script: `confidence_model_v5_experiments.py`
- v5 outputs: `outputs/confidence_model_v5/`

실험 구성:

- E1: audio 512 + text 512 + class one-hot 23 = 1047
- E2: E1 + cosine/L2/dot = 1050
- E3: E2 + OOF prototype 4 + OOF consistency 3 = 1057
- E4: scalar 10 + class one-hot 23 = 33


In [ ]:
# 1. 프로젝트 루트 자동 탐색 및 디렉토리 구조 확인
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'metadata' / 'BSD10k_metadata.csv').exists():
            if (candidate / 'confidence_model_v5_experiments.py').exists():
                return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다. Dcase baseline 폴더 안에서 실행해 주세요.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('\nmetadata:')
for p in sorted((PROJECT_ROOT / 'data' / 'metadata').glob('*')):
    print(' -', p.relative_to(PROJECT_ROOT), f'({p.stat().st_size:,} bytes)')

print('\nfeature dirs:')
for p in sorted((PROJECT_ROOT / 'data' / 'features').iterdir()):
    if p.is_dir():
        print(' -', p.relative_to(PROJECT_ROOT))


In [ ]:
# 2. 노트북 커널 환경 확인
import importlib.util
import sys

print('Python executable:', sys.executable)
print('Python version:', sys.version)

for package in ['numpy', 'pandas', 'torch', 'sklearn', 'scipy', 'matplotlib', 'xgboost']:
    spec = importlib.util.find_spec(package)
    print(f'{package:10s}:', 'OK' if spec else 'NOT FOUND')

try:
    import matplotlib
    print('matplotlib version:', matplotlib.__version__)
    print('matplotlib file:', matplotlib.__file__)
except Exception as exc:
    print('matplotlib import failed:', repr(exc))


In [ ]:
# 3. v5 모듈 import 및 설정 확인
import importlib
import confidence_model_v5_experiments as v5

# 노트북에서 여러 번 실행할 때도 현재 PROJECT_ROOT 기준으로 다시 로드합니다.
os.chdir(PROJECT_ROOT)
v5 = importlib.reload(v5)

print('v5 DEVICE:', v5.DEVICE)
print('output dir:', v5.OUTPUT_DIR)
print('experiments:')
for exp in v5.EXPERIMENTS:
    print(' -', exp['name'], exp['feature_set'], exp['model_type'])

v5.RUN_CONFIG


## 실행 설정

처음에는 아래 설정 그대로 실행하면 E1~E4 전체 5-fold + 최종 full-data 모델 + BSD35k-CS 예측까지 진행합니다.

빠른 smoke test만 하고 싶으면 `folds=2`, `epochs=2`, `train_final_models=False`, `predict_bsd35k=False`로 잠깐 바꿔서 실행하세요.


In [ ]:
# 4. 필요하면 여기서만 설정을 조정하세요.
v5.RUN_CONFIG.update({
    'seed': 42,
    'folds': 5,
    'epochs': 60,
    'patience': 8,
    'batch_size': 256,
    'learning_rate': 8e-4,
    'weight_decay': 1e-4,
    'dropout': 0.30,
    'hidden': [512, 256],
    'tree_estimators': 250,
    'tree_learning_rate': 0.04,
    'history_aggregation': 'pad',
    'use_class_sample_weights': True,
    'class_weight_source': 'metadata',
    'train_final_models': True,
    'predict_bsd35k': True,
})

v5.RUN_CONFIG


In [ ]:
# 5. feature dimension 빠른 확인: 실제 임베딩 로딩 없이 구조만 검증
import numpy as np
import pandas as pd

dummy_parts = {
    'df': pd.DataFrame({'sound_id': ['a', 'b']}),
    'audio': np.zeros((2, 512), dtype=np.float32),
    'text': np.zeros((2, 512), dtype=np.float32),
    'class': np.zeros((2, 23), dtype=np.float32),
    'class_ids': np.array([0, 1]),
    'agreement': np.zeros((2, 3), dtype=np.float32),
}
dummy_scalar = np.zeros((2, 10), dtype=np.float32)

for feature_set in ['E1', 'E2', 'E3', 'E4']:
    x = v5.feature_matrix(dummy_parts, dummy_scalar, feature_set)
    print(feature_set, x.shape)


In [ ]:
# 6. 실제 BSD10k 로딩 및 class/row 수 확인
# 임베딩 .npy 파일을 전부 확인하므로 환경에 따라 몇 분 걸릴 수 있습니다.
parts, class_categories = v5.load_bsd10k_parts()
y_1based = parts['df']['confidence'].to_numpy(dtype=np.int64)

print('rows with embeddings:', len(parts['df']))
print('class categories:', len(class_categories))
print('confidence distribution:')
display(pd.Series(y_1based).value_counts().sort_index().rename('n').to_frame())

zero_scalar = np.zeros((len(parts['df']), 10), dtype=np.float32)
for feature_set in ['E1', 'E2', 'E3', 'E4']:
    print(feature_set, v5.feature_matrix(parts, zero_scalar, feature_set).shape)


## 전체 v5 실행

아래 셀 하나로 E1~E4를 모두 실행합니다.

생성물:

- `outputs/confidence_model_v5/reports/v5_experiment_summary.csv`
- `outputs/confidence_model_v5/reports/v5_fold_metrics.csv`
- `outputs/confidence_model_v5/reports/*_mean_loss_history.csv`
- `outputs/confidence_model_v5/reports/*_confusion_row_normalized.csv`
- `outputs/confidence_model_v5/plots/*_mean_loss.png`
- `outputs/confidence_model_v5/plots/*_confusion_row_normalized.png`
- `outputs/confidence_model_v5/predictions/BSD10k_oof_v5_predictions.csv`
- `outputs/confidence_model_v5/predictions/BSD35k-CS_predicted_v5_*.csv`
- `confidence_model_v5_report_ko.md`


In [ ]:
# 7. 전체 파이프라인 실행
v5.main()


In [ ]:
# 8. 실행 후 결과 요약 확인
summary_path = PROJECT_ROOT / 'outputs' / 'confidence_model_v5' / 'reports' / 'v5_experiment_summary.csv'
fold_path = PROJECT_ROOT / 'outputs' / 'confidence_model_v5' / 'reports' / 'v5_fold_mean_std_summary.csv'

summary_df = pd.read_csv(summary_path)
fold_summary_df = pd.read_csv(fold_path)

display(summary_df)
display(fold_summary_df)


## 노트북 화면에서 바로 확인

아래 셀들은 저장된 결과 파일을 다시 읽어서 노트북 안에 바로 보여줍니다.

- 모델별 train/validation loss curve
- 모델별 row-normalized confusion matrix
- OOF 결과 summary와 예측 분포
- class sample weight 매핑


In [ ]:
# 9. 결과 테이블과 best 모델 확인
from IPython.display import display, Markdown, Image

REPORT_DIR = PROJECT_ROOT / 'outputs' / 'confidence_model_v5' / 'reports'
PLOT_DIR = PROJECT_ROOT / 'outputs' / 'confidence_model_v5' / 'plots'
PRED_DIR = PROJECT_ROOT / 'outputs' / 'confidence_model_v5' / 'predictions'

summary_df = pd.read_csv(REPORT_DIR / 'v5_experiment_summary.csv')
fold_summary_df = pd.read_csv(REPORT_DIR / 'v5_fold_mean_std_summary.csv')
fold_metrics_df = pd.read_csv(REPORT_DIR / 'v5_fold_metrics.csv')

best = summary_df.sort_values(['mae', 'quadratic_weighted_kappa'], ascending=[True, False]).iloc[0]
display(Markdown(
    f"### Best model: `{best['experiment']}`\n"
    f"- MAE: **{best['mae']:.4f}**\n"
    f"- Accuracy: **{best['accuracy'] * 100:.2f}%**\n"
    f"- Macro F1: **{best['macro_f1']:.4f}**\n"
    f"- QWK: **{best['quadratic_weighted_kappa']:.4f}**"
))

display(Markdown('### OOF Summary'))
display(summary_df)

display(Markdown('### Fold Mean ± Std Summary'))
display(fold_summary_df)

display(Markdown('### Fold Metrics'))
display(fold_metrics_df)


In [ ]:
# 10. Train / Validation loss curve를 노트북에 표시
experiment_names = [exp['name'] for exp in v5.EXPERIMENTS]

for exp_name in experiment_names:
    display(Markdown(f'### {exp_name} - Train / Validation Loss'))
    png_path = PLOT_DIR / f'{exp_name}_mean_loss.png'
    csv_path = REPORT_DIR / f'{exp_name}_mean_loss_history.csv'
    if png_path.exists():
        display(Image(filename=str(png_path)))
    elif csv_path.exists():
        hist = pd.read_csv(csv_path)
        display(hist)
        ax = hist.plot(x='epoch', y=['train_loss_mean', 'val_loss_mean'], figsize=(8, 4), grid=True)
        ax.set_title(f'{exp_name} - Fold-mean loss')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss / MAE')
    else:
        print('not found:', png_path, csv_path)


In [ ]:
# 11. 통합 Row-normalized confusion matrix를 노트북에 표시
for exp_name in experiment_names:
    display(Markdown(f'### {exp_name} - Row-normalized Confusion Matrix'))
    png_path = PLOT_DIR / f'{exp_name}_confusion_row_normalized.png'
    csv_path = REPORT_DIR / f'{exp_name}_confusion_row_normalized.csv'
    if png_path.exists():
        display(Image(filename=str(png_path)))
    if csv_path.exists():
        cm_norm = pd.read_csv(csv_path, index_col=0)
        display((cm_norm * 100).round(2))
    else:
        print('not found:', csv_path)


In [ ]:
# 12. OOF prediction 결과와 class sample weight 확인
oof_path = PRED_DIR / 'BSD10k_oof_v5_predictions.csv'
weights_path = REPORT_DIR / 'v5_class_sample_weights.csv'
mapping_path = REPORT_DIR / 'v5_class_distribution_mapping.csv'

if oof_path.exists():
    oof_df = pd.read_csv(oof_path)
    display(Markdown('### OOF Predictions Preview'))
    display(oof_df.head())
    pred_cols = [c for c in oof_df.columns if c.endswith('_class')]
    for col in pred_cols:
        display(Markdown(f'#### {col} distribution'))
        display(oof_df[col].value_counts().sort_index().rename('n').to_frame())

if weights_path.exists():
    display(Markdown('### Metadata Class Sample Weights'))
    display(pd.read_csv(weights_path).sort_values('sample_weight', ascending=False))

if mapping_path.exists():
    display(Markdown('### Supplied Distribution vs Metadata Distribution'))
    display(pd.read_csv(mapping_path))
